In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    precision_recall_curve, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_auc_score, accuracy_score,
    roc_curve
)
import warnings
warnings.filterwarnings('ignore')

print("🎯 TUNING 3.0 - OTIMIZAÇÃO PARA 95% RECALL")
print("🚀 SWEET SPOT: Máxima Detecção + Precisão Otimizada")
print("=" * 70)

# ============================================================================
# 1. CARREGAR E PREPROCESSAR DADOS
# ============================================================================

# Carregar dados
train_path = "/Users/marcelosilva/Desktop/projectOne/6/B- Binary Target DS DT/DatasetTrain.csv"
df_train = pd.read_csv(train_path)

print(f"✅ Dados carregados: {df_train.shape}")

# Configurações
target_col = 'status_nutricional_who'
top_3_features = ['imc_pre_gravidico', 'k06_peso_engravidar', 'h02_peso']

# Limpar dados
exclude_cols = ['id_anon', 'vd_zimc']
df_clean = df_train.drop(columns=exclude_cols, errors='ignore')

# Preprocessamento
df_subset = df_clean[top_3_features + [target_col]].copy()

# Imputar valores faltantes
imputer = SimpleImputer(strategy='median')
df_subset[top_3_features] = imputer.fit_transform(df_subset[top_3_features])

# Separar X e y
X = df_subset[top_3_features]
y = df_subset[target_col]

# Escalar features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"✅ Dataset preprocessado: {X.shape}")
print(f"✅ Target distribution: {y.value_counts().to_dict()}")

total_positives = sum(y)
print(f"✅ Total bebês em risco: {total_positives}")
print(f"✅ Meta 95% recall: detectar {int(total_positives * 0.95)} bebês")

# ============================================================================
# 2. SPLIT ESTRATÉGICO PARA OTIMIZAÇÃO AVANÇADA
# ============================================================================

# Split 70-30 para otimização robusta
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)

print(f"\n✅ Split criado:")
print(f"   Treino: {X_train.shape[0]} amostras")
print(f"   Validação: {X_val.shape[0]} amostras")
print(f"   Positivos validação: {sum(y_val)} (meta: detectar {int(sum(y_val) * 0.95)})")

# ============================================================================
# 3. GRID SEARCH FOCADO EM 95% RECALL
# ============================================================================

print(f"\n🔧 GRID SEARCH OTIMIZADO PARA 95% RECALL")
print("=" * 50)

# Algoritmos otimizados para 95% recall
algorithms_95_recall = {
    'SVC_95': {
        'model': SVC(kernel='linear', probability=True, random_state=42),
        'params': {
            'C': [0.1, 1, 5, 10, 20],  # Range otimizado
            'class_weight': [
                {0: 1, 1: 6},   # Moderado - para 95%
                {0: 1, 1: 8},   # Médio-alto
                {0: 1, 1: 10},  # Alto
                {0: 1, 1: 12},  # Muito alto
            ]
        }
    },
    
    'LogReg_95': {
        'model': LogisticRegression(random_state=42, max_iter=2000),
        'params': {
            'C': [0.1, 1, 5, 10, 20],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear'],
            'class_weight': [
                {0: 1, 1: 6},
                {0: 1, 1: 8},
                {0: 1, 1: 10},
                {0: 1, 1: 12},
            ]
        }
    },
    
    'RF_95': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [5, 7, 9],  # Depth otimizada para 95%
            'min_samples_leaf': [1, 2],
            'class_weight': [
                {0: 1, 1: 6},
                {0: 1, 1: 8},
                {0: 1, 1: 10},
            ]
        }
    }
}

# Executar grid search
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_results = {}

for alg_name, alg_config in algorithms_95_recall.items():
    print(f"\n🔍 Otimizando {alg_name}...")
    
    try:
        # Grid search com recall scoring
        grid_search = GridSearchCV(
            alg_config['model'], 
            alg_config['params'],
            cv=cv_strategy,
            scoring='recall',  # Foco em recall
            n_jobs=-1,
            verbose=0
        )
        
        grid_search.fit(X_train, y_train)
        
        # Salvar resultados
        grid_results[alg_name] = {
            'model': grid_search.best_estimator_,
            'best_params': grid_search.best_params_,
            'cv_recall': grid_search.best_score_
        }
        
        print(f"   ✅ CV Recall: {grid_search.best_score_:.4f}")
        print(f"   🎯 Params: {grid_search.best_params_}")
        
    except Exception as e:
        print(f"   ❌ Erro: {str(e)[:80]}")

# ============================================================================
# 4. THRESHOLD OTIMIZAÇÃO PARA 95% RECALL EXATO
# ============================================================================

def optimize_for_95_recall(model, X_val, y_val, target_recall=0.95, tolerance=0.02):
    """
    Encontra threshold que resulte em exatamente 95% recall
    """
    print(f"\n🎯 OTIMIZAÇÃO PARA {target_recall*100:.0f}% RECALL")
    print(f"Tolerância: ±{tolerance*100:.0f}%")
    
    # Obter probabilidades
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_val)[:, 1]
    else:
        # Para RidgeClassifier, usar decision_function
        from sklearn.calibration import CalibratedClassifierCV
        calibrated_model = CalibratedClassifierCV(model, cv=3)
        calibrated_model.fit(X_train, y_train)
        y_proba = calibrated_model.predict_proba(X_val)[:, 1]
    
    # Testar range de thresholds
    thresholds = np.arange(0.05, 0.8, 0.01)
    
    results = []
    target_found = False
    
    for threshold in thresholds:
        y_pred = (y_proba >= threshold).astype(int)
        
        recall = recall_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred) if sum(y_pred) > 0 else 0
        f1 = f1_score(y_val, y_pred)
        accuracy = accuracy_score(y_val, y_pred)
        
        # Matriz de confusão
        tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
        
        result = {
            'threshold': threshold,
            'recall': recall,
            'precision': precision,
            'f1_score': f1,
            'accuracy': accuracy,
            'tp': tp,
            'fp': fp,
            'fn': fn,
            'tn': tn
        }
        
        results.append(result)
        
        # Verificar se atingiu 95% ± tolerância
        if abs(recall - target_recall) <= tolerance and not target_found:
            print(f"   🎯 THRESHOLD {threshold:.3f}: Recall {recall:.4f} | Precisão {precision:.4f} | F1 {f1:.4f}")
            target_found = True
    
    df_results = pd.DataFrame(results)
    
    # Encontrar melhor resultado próximo a 95%
    target_range = df_results[
        (df_results['recall'] >= target_recall - tolerance) & 
        (df_results['recall'] <= target_recall + tolerance)
    ]
    
    if len(target_range) > 0:
        # Escolher o com melhor F1 score no range
        best_idx = target_range['f1_score'].idxmax()
        best_result = target_range.loc[best_idx]
        
        print(f"\n✅ MELHOR RESULTADO (95% RECALL):")
        print(f"   Threshold: {best_result['threshold']:.3f}")
        print(f"   Recall: {best_result['recall']:.4f}")
        print(f"   Precisão: {best_result['precision']:.4f}")
        print(f"   F1-Score: {best_result['f1_score']:.4f}")
        print(f"   Accuracy: {best_result['accuracy']:.4f}")
        
        return best_result, df_results
    else:
        print(f"   ⚠️ Não conseguiu atingir exatamente 95% recall")
        # Retornar o mais próximo
        df_results['distance'] = abs(df_results['recall'] - target_recall)
        best_idx = df_results['distance'].idxmin()
        best_result = df_results.loc[best_idx]
        
        print(f"   📊 MELHOR APROXIMAÇÃO:")
        print(f"   Recall: {best_result['recall']:.4f} (meta: {target_recall:.3f})")
        print(f"   Threshold: {best_result['threshold']:.3f}")
        
        return best_result, df_results

# Otimizar threshold para cada modelo
threshold_results = {}

for alg_name, result in grid_results.items():
    print(f"\n{'='*20} {alg_name} {'='*20}")
    
    model = result['model']
    best_result, all_results = optimize_for_95_recall(model, X_val, y_val)
    
    threshold_results[alg_name] = {
        'model': model,
        'threshold': best_result['threshold'],
        'metrics': best_result,
        'all_results': all_results
    }

# ============================================================================
# 5. COMPARAÇÃO E SELEÇÃO DO MELHOR MODELO
# ============================================================================

print(f"\n🏆 COMPARAÇÃO FINAL DOS MODELOS (95% RECALL)")
print("=" * 70)

comparison_data = []

for alg_name, result in threshold_results.items():
    metrics = result['metrics']
    
    # Calcular impacto em 849 bebês
    babies_detected = int(849 * metrics['recall'])
    babies_missed = 849 - babies_detected
    
    comparison_data.append({
        'Algoritmo': alg_name,
        'Threshold': f"{result['threshold']:.3f}",
        'Recall': f"{metrics['recall']:.4f}",
        'Precisão': f"{metrics['precision']:.4f}",
        'F1-Score': f"{metrics['f1_score']:.4f}",
        'Accuracy': f"{metrics['accuracy']:.4f}",
        'Bebês_Detectados': babies_detected,
        'Bebês_Perdidos': babies_missed,
        'Falsos_Positivos': metrics['fp']
    })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

# Selecionar melhor modelo
best_model_name = comparison_df.loc[comparison_df['F1-Score'].astype(float).idxmax(), 'Algoritmo']
best_model_result = threshold_results[best_model_name]

print(f"\n🥇 MODELO CAMPEÃO: {best_model_name}")
print(f"   Threshold: {best_model_result['threshold']:.3f}")
print(f"   Performance: R={best_model_result['metrics']['recall']:.3f} | P={best_model_result['metrics']['precision']:.3f} | F1={best_model_result['metrics']['f1_score']:.3f}")

# ============================================================================
# 6. VISUALIZAÇÕES COMPARATIVAS
# ============================================================================

def create_95_recall_visualizations():
    """
    Visualizações específicas para análise de 95% recall
    """
    print(f"\n📊 Criando visualizações...")
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Gráfico 1: Comparação de modelos
    models = list(comparison_df['Algoritmo'])
    recalls = [float(x) for x in comparison_df['Recall']]
    precisions = [float(x) for x in comparison_df['Precisão']]
    f1s = [float(x) for x in comparison_df['F1-Score']]
    
    x = np.arange(len(models))
    width = 0.25
    
    ax1.bar(x - width, recalls, width, label='Recall', color='blue', alpha=0.7)
    ax1.bar(x, precisions, width, label='Precisão', color='red', alpha=0.7)
    ax1.bar(x + width, f1s, width, label='F1-Score', color='green', alpha=0.7)
    
    ax1.set_xlabel('Modelos')
    ax1.set_ylabel('Métrica')
    ax1.set_title('Comparação de Modelos (95% Recall)')
    ax1.set_xticks(x)
    ax1.set_xticklabels(models, rotation=45)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.axhline(y=0.95, color='blue', linestyle='--', alpha=0.5, label='Meta Recall')
    
    # Gráfico 2: Impacto médico
    babies_detected = [int(x) for x in comparison_df['Bebês_Detectados']]
    babies_missed = [int(x) for x in comparison_df['Bebês_Perdidos']]
    
    ax2.bar(models, babies_detected, label='Detectados', color='green', alpha=0.7)
    ax2.bar(models, babies_missed, bottom=babies_detected, label='Perdidos', color='red', alpha=0.7)
    
    ax2.set_xlabel('Modelos')
    ax2.set_ylabel('Número de Bebês')
    ax2.set_title('Impacto Médico (dos 849 bebês em risco)')
    ax2.legend()
    ax2.set_xticklabels(models, rotation=45)
    
    # Gráfico 3: Falsos positivos vs Precisão
    false_positives = [int(x) for x in comparison_df['Falsos_Positivos']]
    
    ax3.scatter(precisions, false_positives, s=100, alpha=0.7)
    for i, model in enumerate(models):
        ax3.annotate(model, (precisions[i], false_positives[i]), 
                    xytext=(5, 5), textcoords='offset points')
    
    ax3.set_xlabel('Precisão')
    ax3.set_ylabel('Falsos Positivos')
    ax3.set_title('Trade-off: Precisão vs Falsos Positivos')
    ax3.grid(True, alpha=0.3)
    
    # Gráfico 4: Threshold sensitivity para melhor modelo
    if best_model_name in threshold_results:
        best_results = threshold_results[best_model_name]['all_results']
        
        ax4.plot(best_results['threshold'], best_results['recall'], 'b-', label='Recall')
        ax4.plot(best_results['threshold'], best_results['precision'], 'r-', label='Precisão')
        ax4.plot(best_results['threshold'], best_results['f1_score'], 'g-', label='F1-Score')
        ax4.axhline(y=0.95, color='blue', linestyle='--', alpha=0.5)
        ax4.axvline(x=best_model_result['threshold'], color='black', linestyle='--', alpha=0.5)
        
        ax4.set_xlabel('Threshold')
        ax4.set_ylabel('Métrica')
        ax4.set_title(f'Sensitivity Analysis - {best_model_name}')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return fig

# Criar visualizações
viz_fig = create_95_recall_visualizations()

# ============================================================================
# 7. INTERPRETAÇÃO CLÍNICA DETALHADA
# ============================================================================

print(f"\n👨‍⚕️ INTERPRETAÇÃO CLÍNICA - 95% RECALL")
print("=" * 50)

best_metrics = best_model_result['metrics']
babies_detected_95 = int(849 * best_metrics['recall'])
babies_missed_95 = 849 - babies_detected_95

print(f"🎯 MODELO OTIMIZADO ({best_model_name}):")
print(f"   Threshold: {best_model_result['threshold']:.3f}")
print(f"   Recall: {best_metrics['recall']:.4f} ({best_metrics['recall']*100:.1f}%)")
print(f"   Precisão: {best_metrics['precision']:.4f} ({best_metrics['precision']*100:.1f}%)")
print(f"   F1-Score: {best_metrics['f1_score']:.4f}")

print(f"\n👶 IMPACTO MÉDICO REAL:")
print(f"   ✅ Bebês detectados: {babies_detected_95}/849 ({best_metrics['recall']*100:.1f}%)")
print(f"   ❌ Bebês perdidos: {babies_missed_95} ({(1-best_metrics['recall'])*100:.1f}%)")
print(f"   ⚠️ Falsos alarmes: {best_metrics['fp']}")
print(f"   📊 A cada 100 alertas: {best_metrics['precision']*100:.0f} são casos reais")

# Comparação com 99% recall
print(f"\n⚖️ COMPARAÇÃO: 95% vs 99% RECALL")
print(f"   Bebês perdidos: {babies_missed_95} vs 7 (diferença: +{babies_missed_95-7})")
print(f"   Precisão estimada: {best_metrics['precision']*100:.1f}% vs 25% (melhoria: +{(best_metrics['precision']-0.25)*100:.0f}%)")
print(f"   Falsos alarmes: Redução significativa esperada")

# Recomendação clínica
if best_metrics['precision'] >= 0.35 and best_metrics['recall'] >= 0.94:
    clinical_recommendation = "🏆 EXCELENTE para implementação clínica"
elif best_metrics['precision'] >= 0.30 and best_metrics['recall'] >= 0.93:
    clinical_recommendation = "✅ MUITO BOM para triagem populacional"
elif best_metrics['precision'] >= 0.25 and best_metrics['recall'] >= 0.90:
    clinical_recommendation = "👍 BOM para medicina preventiva"
else:
    clinical_recommendation = "⚠️ Precisa de mais otimização"

print(f"\n🩺 RECOMENDAÇÃO CLÍNICA: {clinical_recommendation}")

# ============================================================================
# 8. CONFIGURAÇÃO FINAL PARA DEPLOY
# ============================================================================

print(f"\n🚀 CONFIGURAÇÃO FINAL PARA DEPLOY")
print("=" * 50)

# Parâmetros finais
final_deploy_config = {
    'model_type': best_model_name,
    'model_params': grid_results[best_model_name]['best_params'],
    'features': top_3_features,
    'optimal_threshold': best_model_result['threshold'],
    'preprocessing': {
        'scaler_mean': scaler.mean_.tolist(),
        'scaler_scale': scaler.scale_.tolist(),
        'imputer_strategy': 'median'
    },
    'performance_metrics': {
        'recall': best_metrics['recall'],
        'precision': best_metrics['precision'],
        'f1_score': best_metrics['f1_score'],
        'accuracy': best_metrics['accuracy']
    },
    'medical_impact': {
        'babies_detected_849': babies_detected_95,
        'babies_missed': babies_missed_95,
        'false_positives': int(best_metrics['fp']),
        'detection_rate_percent': best_metrics['recall'] * 100
    }
}

print(f"💾 CONFIGURAÇÃO OTIMIZADA (95% RECALL):")
print(f"   Algoritmo: {final_deploy_config['model_type']}")
print(f"   Threshold: {final_deploy_config['optimal_threshold']:.3f}")
print(f"   Features: {final_deploy_config['features']}")
print(f"   Performance: R={final_deploy_config['performance_metrics']['recall']:.3f} P={final_deploy_config['performance_metrics']['precision']:.3f}")
print(f"   Impacto: {final_deploy_config['medical_impact']['babies_detected_849']}/849 bebês detectados")

print(f"\n📋 PRÓXIMOS PASSOS:")
print(f"1. ✅ Modelo otimizado para 95% recall")
print(f"2. 🧪 Validar no dataset de teste")
print(f"3. 🌐 Implementar no site se validação for bem-sucedida")
print(f"4. 📄 Escrever paper com resultados otimizados")

print(f"\n🎯 OTIMIZAÇÃO 95% RECALL CONCLUÍDA!")
print(f"💪 Sweet spot encontrado: {best_metrics['recall']*100:.1f}% recall + {best_metrics['precision']*100:.1f}% precisão!")